# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [ ]:
context = [ {'role':'system', 'content': """
You are a SQL generation assistant. Use the following database schema to answer user questions. 
All answers must start with "this is your SQL", followed by the SQL code, and a brief explanation.

### Database Schema:

CREATE TABLE employees (
  ID_usr INT PRIMARY KEY,
  name VARCHAR(100)
);
/*
Sample rows:
(1, 'John Doe'),
(2, 'Jane Smith'),
(3, 'Robert Brown')
*/

CREATE TABLE salary (
  ID_usr INT,
  year DATE,
  salary FLOAT,
  FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);
/*
Sample rows:
(1, '2023-01-01', 50000.0),
(2, '2023-01-01', 65000.0),
(1, '2024-01-01', 52000.0)
*/

CREATE TABLE studies (
  ID INT PRIMARY KEY,
  ID_usr INT,
  educational_level INT,
  Institution VARCHAR(150),
  Years DATE,
  Speciality VARCHAR(100),
  FOREIGN KEY (ID_usr) REFERENCES employees(ID_usr)
);
/*
Sample rows:
(101, 1, 3, 'Vilnius University', '2020-06-15', 'Computer Science'),
(102, 2, 4, 'MIT', '2018-05-20', 'Physics')
*/

### Few-Shot Samples:

Question: "Show the name of the employee with the highest salary in 2023."
SQL: SELECT e.name FROM employees e JOIN salary s ON e.ID_usr = s.ID_usr WHERE s.year = '2023-01-01' ORDER BY s.salary DESC LIMIT 1;

Question: "List all institutions where John Doe studied."
SQL: SELECT st.Institution FROM studies st JOIN employees e ON st.ID_usr = e.ID_usr WHERE e.name = 'John Doe';

Question: "How many employees have a degree in Computer Science?"
SQL: SELECT COUNT(DISTINCT ID_usr) FROM studies WHERE Speciality = 'Computer Science';
"""} ]

In [5]:
context.append( {'role':'system', 'content':"""
-- Maintain the SQL order simple and efficient as you can, using valid SQLite.
-- Here are examples of how you should answer:

Question: "Show me the names of all employees."
this is your SQL
SELECT name FROM employees;
This query selects all values from the 'name' column in the 'employees' table.

Question: "What is the total salary paid in 2023?"
this is your SQL
SELECT SUM(salary) FROM salary WHERE year LIKE '2023%';
This query calculates the sum of the 'salary' column for all entries where the year starts with 2023.

Question: "List institutions for employees with a salary over 60000."
this is your SQL
SELECT st.Institution 
FROM studies st 
JOIN salary sa ON st.ID_usr = sa.ID_usr 
WHERE sa.salary > 60000;
This query joins the 'studies' and 'salary' tables by user ID to find institutions linked to high salaries.
"""
})

In [6]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [8]:
#new
context_user = context.copy()
print(return_CCRMSQL("""List the name and speciality of all employees who studied at Vilnius University.""", context_user))

this is your SQL
SELECT e.name, st.Speciality 
FROM employees e 
JOIN studies st ON e.ID_usr = st.ID_usr 
WHERE st.Institution = 'Vilnius University';
This query retrieves the name and speciality of employees who studied at Vilnius University by joining the 'employees' and 'studies' tables on the user ID and filtering for the specified institution.


In [12]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What is the average salary of employees who have a 'Computer Science' speciality?", old_context_user))

This is your SQL:
```sql
SELECT AVG(s.salary) AS average_salary
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE st.Speciality = 'Computer Science';
```

This SQL query retrieves the average salary of employees who have a 'Computer Science' speciality. It joins the employees, studies, and salary tables on the employee ID, filters the results to only include employees with a 'Computer Science' speciality, and calculates the average salary for those employees.


In [13]:
#new
print(return_CCRMSQL("What is the average salary of employees who have a 'Computer Science' speciality?", context_user))

this is your SQL
SELECT AVG(s.salary) 
FROM salary s 
JOIN studies st ON s.ID_usr = st.ID_usr 
WHERE st.Speciality = 'Computer Science';
This query calculates the average salary of employees who have a 'Computer Science' speciality by joining the 'salary' and 'studies' tables on user ID and filtering for the specific speciality.


In [14]:
#old
print(return_CCRMSQL("Show the average salary for each year, but only for employees who have more than 5 years of studies.", old_context_user))

This is your SQL:
```sql
SELECT s.year, AVG(s.salary) AS average_salary
FROM salary s
JOIN studies st ON s.ID_usr = st.ID_usr
WHERE st.Years > 5
GROUP BY s.year;
```

This SQL query retrieves the average salary for each year from the `salary` table, considering only employees who have more than 5 years of studies. It achieves this by joining the `salary` table with the `studies` table on the `ID_usr` column, filtering the results based on the condition `st.Years > 5`, and then calculating the average salary for each year using the `AVG` function.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [15]:
print(return_CCRMSQL("dentify the names of employees who have studied for more than 4 years and earn a salary above the average of all salaries in 2023.", context_user))

this is your SQL
SELECT e.name
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE st.Years < DATE('now') AND st.Years > DATE('now', '-4 years')
AND s.salary > (SELECT AVG(salary) FROM salary WHERE year = '2023-01-01');
This query retrieves the names of employees who have studied for more than 4 years and earn a salary above the average of all salaries in 2023.


In [16]:
print(return_CCRMSQL("Send a congratulatory email to the employee with ID 1 for finishing their PhD in 2024.", context_user))

this is your SQL
SELECT name, email
FROM employees
WHERE ID_usr = 1;
This query selects the name and email of the employee with ID 1. However, the schema provided does not include an 'email' column. If you have an 'email' column in the 'employees' table, please provide the column name for a more accurate query.


In [17]:
print(return_CCRMSQL("List the total salary increase for each employee between 2023 and 2024", context_user))

this is your SQL
SELECT e.name, (s2.salary - s1.salary) AS salary_increase
FROM employees e
JOIN salary s1 ON e.ID_usr = s1.ID_usr AND s1.year = '2023-01-01'
JOIN salary s2 ON e.ID_usr = s2.ID_usr AND s2.year = '2024-01-01';
This query calculates the salary increase for each employee between the years 2023 and 2024 by joining the 'employees' table with the 'salary' table twice, once for each year, and then calculating the difference in salary.
